<a href="https://colab.research.google.com/github/wu-warren/BIGML_Application/blob/main/Synthetic%20Data%20Generation/Part_1_C161_GPT2_Generation_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://arxiv.org/abs/2410.21717

In [ ]:
# !pip install -U transformers

## Local Inference on GPU
Model page: https://huggingface.co/openai-community/gpt2

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/openai-community/gpt2)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# # Use a pipeline as a high-level helper
# from transformers import pipeline

# pipe = pipeline("text-generation", model="openai-community/gpt2")

In [ ]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
# model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

Fine-tuning: Encoding to sentences

In [ ]:
import pandas as pd
from typing import List, Optional, Iterable

# def format_value(val):
#     """Format values nicely for text."""
#     if pd.isna(val):
#         return "unknown"
#     if isinstance(val, float):
#         return f"{val:.3f}".rstrip('0').rstrip('.')  # e.g. 1.23, 10
#     return str(val)

# def format_feature_name(col: str) -> str:
#     """Make column names more human-readable."""
#     return col.replace('_', ' ').lower()

def row_to_sentence(
    row: pd.Series,
    feature_cols: List[str],
    target_col: Optional[str] = None,
    include_target: bool = True,
) -> str:
    parts = []

    # features
    for col in feature_cols:
        val = row[col]
        feature_text = col
        val_text = val
        parts.append(f"{feature_text} is {val_text}")

    # target / label
    if include_target and target_col is not None:
        y = row[target_col]
        y_text = (y)
        parts.append(f"target is {y_text}")

    sentence = ", ".join(parts) + "."
    return sentence

def encode_dataset_to_sentences(
    df: pd.DataFrame,
    feature_cols: List[str],
    target_col: Optional[str] = None,
    include_target: bool = True,
) -> Iterable[str]:
    for _, row in df.iterrows():
        yield row_to_sentence(row, feature_cols, target_col, include_target)


In [ ]:
import pandas as pd

# Load the full dataset
full_df = pd.read_csv("real_train_ctgan_200k_safe27.csv")

# Randomly sample 1000 rows
df = full_df.sample(n=1000, random_state=42) # Using random_state for reproducibility

# Display the first 5 rows of the sampled DataFrame
display(df.head())

,creat_type_cd,f_cat_uniq,f_refresh_sum,slot_id,f_rows,f_up_sum,f_dislike_sum,f_refresh_mean,u_refreshTimes,u_newsCatInterestsST_len,...,adv_id,task_id,inter_type_cd,hispace_app_tags,spread_app_id,app_second_class,ad_click_list_v002_uniq,ad_click_list_v002_len,f_hour_cos,label
119737,5,10,0,53,16,132,75,0.0,0,5,...,17828,18800,4,43,312,18,2,2,-0.762527,0
72272,8,13,102,35,34,221,25,3.0,3,5,...,11883,29699,5,23,283,17,5,5,-0.076924,0
158154,7,29,205,59,41,241,128,5.0,5,5,...,17440,32607,3,19,175,18,3,3,-0.379732,0
65426,8,27,384,59,64,248,133,6.0,6,5,...,13258,13860,5,47,246,14,5,5,-0.849202,0
30074,8,22,205,63,41,267,116,5.0,5,5,...,11035,33709,5,39,350,15,3,3,-0.953396,0


In [ ]:
df.describe()

,creat_type_cd,f_cat_uniq,f_refresh_sum,slot_id,f_rows,f_up_sum,f_dislike_sum,f_refresh_mean,u_refreshTimes,u_newsCatInterestsST_len,...,adv_id,task_id,inter_type_cd,hispace_app_tags,spread_app_id,app_second_class,ad_click_list_v002_uniq,ad_click_list_v002_len,f_hour_cos,label
count,1000.0000,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,7.9660,34.303000,654.27700,31.692000,96.366000,323.224000,170.033000,5.750565,5.737000,4.778000,...,16574.236000,23230.566000,4.252000,36.655000,232.550000,18.070000,4.245000,4.245000,-0.613614,0.019000
std,1.8959,21.053038,827.52624,17.186347,96.052038,241.595389,136.217551,2.932794,2.945591,0.908593,...,3968.532092,8086.931531,0.644136,13.161048,73.074287,4.367399,1.290209,1.290209,0.432852,0.136593
min,2.0000,1.000000,0.00000,12.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10015.000000,10073.000000,3.000000,12.000000,101.000000,13.000000,1.000000,1.000000,-1.000000,0.000000
25%,8.0000,18.000000,109.50000,16.000000,32.000000,151.000000,73.000000,4.000000,4.000000,5.000000,...,12740.000000,15655.750000,4.000000,20.000000,162.000000,14.000000,4.000000,4.000000,-0.954580,0.000000
50%,8.0000,31.000000,405.50000,26.000000,69.000000,270.000000,139.000000,6.000000,6.000000,5.000000,...,16758.500000,22795.500000,4.000000,43.000000,213.000000,17.000000,5.000000,5.000000,-0.765859,0.000000
75%,10.0000,48.000000,855.00000,50.000000,125.000000,438.250000,231.250000,8.000000,8.000000,5.000000,...,19903.750000,30797.500000,5.000000,47.000000,309.000000,22.000000,5.000000,5.000000,-0.422618,0.000000
max,10.0000,124.000000,7488.00000,69.000000,832.000000,1870.000000,1047.000000,9.000000,9.000000,5.000000,...,23517.000000,36342.000000,5.000000,53.000000,372.000000,30.000000,5.000000,5.000000,1.000000,1.000000


In [ ]:
df['label'] = df['label'].astype(bool)
target_col = df.columns[-1]
feature_cols = [c for c in df.columns if c != target_col]

In [ ]:
encoded_df = encode_dataset_to_sentences(df, feature_cols, target_col, include_target=True)

In [ ]:
list(encoded_df)[:5]

['creat_type_cd is 5, f_cat_uniq is 10, f_refresh_sum is 0, slot_id is 53, f_rows is 16, f_up_sum is 132, f_dislike_sum is 75, f_refresh_mean is 0.0, u_refreshTimes is 0, u_newsCatInterestsST_len is 5, f_up_mean is 8.25, u_feedLifeCycle is 17, u_newsCatInterestsST_uniq is 5, f_entities_len_mean is 4.875, f_dislike_mean is 4.6875, f_browser_life is 17.0, adv_prim_id is 1036, device_size is 2117, adv_id is 17828, task_id is 18800, inter_type_cd is 4, hispace_app_tags is 43, spread_app_id is 312, app_second_class is 18, ad_click_list_v002_uniq is 2, ad_click_list_v002_len is 2, f_hour_cos is -0.7625272039063882, target is False.',
 'creat_type_cd is 8, f_cat_uniq is 13, f_refresh_sum is 102, slot_id is 35, f_rows is 34, f_up_sum is 221, f_dislike_sum is 25, f_refresh_mean is 3.0, u_refreshTimes is 3, u_newsCatInterestsST_len is 5, f_up_mean is 6.5, u_feedLifeCycle is 17, u_newsCatInterestsST_uniq is 4, f_entities_len_mean is 4.941176470588236, f_dislike_mean is 0.7352941176470589, f_brows

Fine-tuning: Permuting predictor variables

In [ ]:
import random

def row_to_sentence_with_permutation(
    row: pd.Series,
    feature_cols: List[str],
    target_col: Optional[str] = None,
    include_target: bool = True,
    permute_features: bool = False,
) -> str:
    cols = feature_cols.copy()
    if permute_features:
        random.shuffle(cols)

    parts = []
    for col in cols:
        val = row[col]
        feature_text = col
        val_text = val
        parts.append(f"{feature_text} is {val_text}")

    if include_target and target_col is not None:
        y = row[target_col]
        y_text = y
        parts.append(f"target is {y_text}")

    return ", ".join(parts) + "."

def build_Dprime_real(df, feature_cols, target_col):
    sentences = []
    for _, row in df.iterrows():
        s  = row_to_sentence_with_permutation(row, feature_cols, target_col,
                                              include_target=True,
                                              permute_features=False)
        s_prime = row_to_sentence_with_permutation(row, feature_cols, target_col,
                                                   include_target=True,
                                                   permute_features=True)
        sentences.append(s)
        sentences.append(s_prime)
    return sentences


In [ ]:
Dprime = build_Dprime_real(df, feature_cols, target_col)

In [ ]:
Dprime[:5]

['creat_type_cd is 5, f_cat_uniq is 10, f_refresh_sum is 0, slot_id is 53, f_rows is 16, f_up_sum is 132, f_dislike_sum is 75, f_refresh_mean is 0.0, u_refreshTimes is 0, u_newsCatInterestsST_len is 5, f_up_mean is 8.25, u_feedLifeCycle is 17, u_newsCatInterestsST_uniq is 5, f_entities_len_mean is 4.875, f_dislike_mean is 4.6875, f_browser_life is 17.0, adv_prim_id is 1036, device_size is 2117, adv_id is 17828, task_id is 18800, inter_type_cd is 4, hispace_app_tags is 43, spread_app_id is 312, app_second_class is 18, ad_click_list_v002_uniq is 2, ad_click_list_v002_len is 2, f_hour_cos is -0.7625272039063882, target is False.',
 'f_browser_life is 17.0, f_dislike_sum is 75, creat_type_cd is 5, f_up_sum is 132, u_feedLifeCycle is 17, f_cat_uniq is 10, inter_type_cd is 4, u_refreshTimes is 0, ad_click_list_v002_len is 2, app_second_class is 18, u_newsCatInterestsST_uniq is 5, f_refresh_sum is 0, task_id is 18800, u_newsCatInterestsST_len is 5, f_hour_cos is -0.7625272039063882, hispace_a

Fine-Tuning GPT2

In [ ]:
pip install -U transformers datasets accelerate


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 23.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
import torch


In [ ]:
dataset = Dataset.from_dict({"text": Dprime})

# Optionally split train/validation
dataset = dataset.train_test_split(test_size=0.05, shuffle=True, seed=42)
train_ds = dataset["train"]
eval_ds  = dataset["test"]


In [ ]:
model_name = "gpt2"  # or "gpt2-medium", etc.

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
max_length = 256  # or larger if your sentences are long

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )

tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized_eval  = eval_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

tokenized_train.set_format(type="torch")
tokenized_eval.set_format(type="torch")


Map:   0%|          | 0/1900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

# In case vocab size changed due to adding pad token or custom tokens
model.resize_token_embeddings(len(tokenizer))


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Embedding(50257, 768)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,          # this makes it auto-regressive next-token prediction
)


In [ ]:
training_args = TrainingArguments(
    output_dir="./gpt2-pred-llm",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    logging_steps=100,
    save_strategy="epoch",
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    fp16=torch.cuda.is_available(),
    report_to="none",        # disable wandb/etc
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)


In [ ]:
trainer.train()
trainer.save_model("./gpt2-pred-llm")
tokenizer.save_pretrained("./gpt2-pred-llm")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.389500,0.802057
2,0.814900,0.755971
3,0.766100,0.742327


('./gpt2-pred-llm/tokenizer_config.json',
 './gpt2-pred-llm/special_tokens_map.json',
 './gpt2-pred-llm/vocab.json',
 './gpt2-pred-llm/merges.txt',
 './gpt2-pred-llm/added_tokens.json',
 './gpt2-pred-llm/tokenizer.json')

In [ ]:
import os

print(os.getcwd())
print(os.listdir("."))

print(os.listdir("./gpt2-pred-llm"))

/content
['.config', 'real_train_ctgan_200k_safe27.csv', 'gpt2-pred-llm', 'sample_data']
['tokenizer_config.json', 'training_args.bin', 'checkpoint-119', 'config.json', 'checkpoint-238', 'vocab.json', 'model.safetensors', 'tokenizer.json', 'merges.txt', 'special_tokens_map.json', 'generation_config.json', 'checkpoint-357']


In [ ]:
import shutil
from google.colab import files

model_file_path = './gpt2-pred-llm/model.safetensors'

print(f"Downloading '{model_file_path}'...")

# Download the specific model file
files.download(model_file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Sampling Step

In [ ]:
model_dir = "./gpt2-pred-llm"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir)  # will use model.safetensors

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [ ]:
def Q(condition: str, n: int, max_new_tokens: int = 64) -> list[str]:
    inputs = tokenizer(
        [condition] * n,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=1.0,
            top_p=0.95,
            top_k=50,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.batch_decode(out, skip_special_tokens=True)


In [ ]:
def parse_sentence_to_features(sentence, feature_cols):
    sentence = sentence.strip().rstrip(".")
    clauses = [c.strip() for c in sentence.split(",") if c.strip()]

    result = {col: None for col in feature_cols}

    for clause in clauses:
        if " is " not in clause:
            continue
        key, val = clause.split(" is ", 1)
        key, val = key.strip(), val.strip()

        if key not in result:
            continue

        if val.lower() in ("true", "false"):
            result[key] = val.lower() == "true"
        else:
            try:
                f = float(val)
                result[key] = int(f) if f.is_integer() else f
            except:
                result[key] = val

    return result


In [ ]:
import numpy as np

def sample_value_from_column(df, col):
    vals = df[col].dropna().values
    return np.random.choice(vals)


In [ ]:
import pandas as pd

def sampling_phase(df_real, feature_cols, N=1000):
    M = len(feature_cols)
    n_per_feature = max(1, N // M)

    synthetic_rows = []

    for col in feature_cols:
        v = sample_value_from_column(df_real, col)
        condition = f"{col} is {v}, "                       # partial prompt
        gens = Q(condition, n=n_per_feature, max_new_tokens=512)

        for s in gens:
            synthetic_rows.append(parse_sentence_to_features(s, feature_cols))

    # Ensure we return exactly N
    synthetic_rows = synthetic_rows[:N]

    return pd.DataFrame(synthetic_rows, columns=feature_cols)

# RUN SAMPLING PHASE
feature_cols = [c for c in df.columns if c not in ("label", "target")]
D_fake_X = sampling_phase(df, feature_cols, N=1000)

print(D_fake_X.head())
print(D_fake_X.shape)


   creat_type_cd  f_cat_uniq  f_refresh_sum  slot_id  f_rows  f_up_sum  \
0            8.0        38.0          914.0     16.0   154.0       528   
1            8.0        91.0         1028.0     58.0   105.0       569   
2            8.0       253.0         2464.0     25.0   190.0      1197   
3            8.0        50.0         1076.0     13.0   149.0       979   
4            8.0        48.0         2474.0     33.0   195.0      1551   

   f_dislike_sum  f_refresh_mean  u_refreshTimes  u_newsCatInterestsST_len  \
0          516.0             9.0             9.0                       5.0   
1          371.0             7.0             7.0                       5.0   
2          703.0             9.0             9.0                       5.0   
3          495.0             8.0             8.0                       5.0   
4         1390.0             9.0             9.0                       5.0   

   ...  device_size  adv_id  task_id  inter_type_cd  hispace_app_tags  \
0  ...       

Querying Step

In [ ]:
def features_to_prompt_for_target(row, feature_cols):
    """
    Build a prompt that mirrors the training sentences in D′:
    'col1 is v1, col2 is v2, ..., f_hour_cos is vM, target is '
    """
    parts = []
    for col in feature_cols:
        val = row[col]
        parts.append(f"{col} is {val}")
    # Note the space after 'is' to make parsing easier
    prompt = ", ".join(parts) + ", target is "
    return prompt


In [ ]:
feature_cols = [c for c in D_fake_X.columns if c not in ("target", "label")]


In [ ]:
def Q(prompt: str, n: int = 1, max_new_tokens: int = 4):
    inputs = tokenizer(
        [prompt] * n,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # just enough for "False." / "True."
            do_sample=True,                 # sampling = p(ŷ | x̂)
            top_k=10,
            top_p=0.9,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.batch_decode(out, skip_special_tokens=True)


In [ ]:
import re

def parse_target_bool(sentence: str):
    """
    Extract True/False from text like:
    '..., target is False.' or '..., target is True'
    """
    m = re.search(r'target\s+is\s+([^\s,\.]+)', sentence, flags=re.IGNORECASE)
    if not m:
        return None

    token = m.group(1).lower()
    if token == "true":
        return True
    if token == "false":
        return False
    return None  # anything else is treated as invalid


In [ ]:
def querying_phase(D_fake_X, feature_cols):
    # make sure we don't include label/target in the features
    feature_cols = [c for c in feature_cols if c not in ("label", "target")]

    y_hats = []
    for _, row in D_fake_X.iterrows():
        prompt = features_to_prompt_for_target(row, feature_cols)
        generated = Q(prompt, n=1, max_new_tokens=4)[0]
        yhat = parse_target_bool(generated)
        y_hats.append(yhat)

    result = D_fake_X.copy()
    result["target"] = y_hats
    return result


In [ ]:
row = D_fake_X.iloc[0]
prompt = features_to_prompt_for_target(row, feature_cols)
print("PROMPT:\n", prompt)

generated = Q(prompt, n=1, max_new_tokens=4)[0]
print("\nGENERATED:\n", generated)

print("\nPARSED TARGET:", parse_target_bool(generated))


PROMPT:
 creat_type_cd is 8.0, f_cat_uniq is 38.0, f_refresh_sum is 914.0, slot_id is 16.0, f_rows is 154.0, f_up_sum is 528, f_dislike_sum is 516.0, f_refresh_mean is 9.0, u_refreshTimes is 9.0, u_newsCatInterestsST_len is 5.0, f_up_mean is 5.673333333333333, u_feedLifeCycle is 17.0, u_newsCatInterestsST_uniq is 5.0, f_entities_len_mean is 4.86666666666665, f_dislike_mean is 1.353535353535365, f_browser_life is 17.0, adv_prim_id is 2066.0, device_size is 3103.0, adv_id is 12382, task_id is 29332.0, inter_type_cd is 4.0, hispace_app_tags is 20.0, spread_app_id is 213.0, app_second_class is 18.0, ad_click_list_v002_uniq is 5.0, ad_click_list_v002_len is 5.0, f_hour_cos is -0.9979581612609536, target is 

GENERATED:
 creat_type_cd is 8.0, f_cat_uniq is 38.0, f_refresh_sum is 914.0, slot_id is 16.0, f_rows is 154.0, f_up_sum is 528, f_dislike_sum is 516.0, f_refresh_mean is 9.0, u_refreshTimes is 9.0, u_newsCatInterestsST_len is 5.0, f_up_mean is 5.673333333333333, u_feedLifeCycle is 17.0

In [ ]:
row = D_fake_X.iloc[0]
prompt = features_to_prompt_for_target(row, feature_cols)
print("PROMPT:\n", prompt)

generated = Q(prompt, n=1, max_new_tokens=4)[0]
print("\nGENERATED:\n", generated)


PROMPT:
 creat_type_cd is 8.0, f_cat_uniq is 38.0, f_refresh_sum is 914.0, slot_id is 16.0, f_rows is 154.0, f_up_sum is 528, f_dislike_sum is 516.0, f_refresh_mean is 9.0, u_refreshTimes is 9.0, u_newsCatInterestsST_len is 5.0, f_up_mean is 5.673333333333333, u_feedLifeCycle is 17.0, u_newsCatInterestsST_uniq is 5.0, f_entities_len_mean is 4.86666666666665, f_dislike_mean is 1.353535353535365, f_browser_life is 17.0, adv_prim_id is 2066.0, device_size is 3103.0, adv_id is 12382, task_id is 29332.0, inter_type_cd is 4.0, hispace_app_tags is 20.0, spread_app_id is 213.0, app_second_class is 18.0, ad_click_list_v002_uniq is 5.0, ad_click_list_v002_len is 5.0, f_hour_cos is -0.9979581612609536, target is 

GENERATED:
 creat_type_cd is 8.0, f_cat_uniq is 38.0, f_refresh_sum is 914.0, slot_id is 16.0, f_rows is 154.0, f_up_sum is 528, f_dislike_sum is 516.0, f_refresh_mean is 9.0, u_refreshTimes is 9.0, u_newsCatInterestsST_len is 5.0, f_up_mean is 5.673333333333333, u_feedLifeCycle is 17.0

In [ ]:
feature_cols = [c for c in D_fake_X.columns if c not in ("label", "target")]
D_fake_full = querying_phase(D_fake_X, feature_cols)

print(D_fake_full.head())
print(D_fake_full["target"].value_counts())


   creat_type_cd  f_cat_uniq  f_refresh_sum  slot_id  f_rows  f_up_sum  \
0            8.0        38.0          914.0     16.0   154.0       528   
1            8.0        91.0         1028.0     58.0   105.0       569   
2            8.0       253.0         2464.0     25.0   190.0      1197   
3            8.0        50.0         1076.0     13.0   149.0       979   
4            8.0        48.0         2474.0     33.0   195.0      1551   

   f_dislike_sum  f_refresh_mean  u_refreshTimes  u_newsCatInterestsST_len  \
0          516.0             9.0             9.0                       5.0   
1          371.0             7.0             7.0                       5.0   
2          703.0             9.0             9.0                       5.0   
3          495.0             8.0             8.0                       5.0   
4         1390.0             9.0             9.0                       5.0   

   ...  adv_id  task_id  inter_type_cd  hispace_app_tags  spread_app_id  \
0  ...   12

Conditional sampling on target variable does not work --> let's try sampling the entire row at once

In [ ]:
import pandas as pd

def sampling_phase_joint(df_real, feature_cols, N=1000,
                         max_new_tokens=512,
                         batch_size=16):
    """
    Jointly sample X and target from the fine-tuned GPT-2.

    df_real: real df (we only need it for column names)
    feature_cols: columns to generate, INCLUDING 'target'
    N: number of synthetic rows
    """

    first_col = feature_cols[0]  # e.g. 'creat_type_cd'
    synthetic_rows = []

    while len(synthetic_rows) < N:
        cur_n = min(batch_size, N - len(synthetic_rows))

        # Simple prefix to steer generation into the row pattern
        # You can also try "" or f"{first_col} is "
        prompt = f"{first_col} is"
        gens = Q(prompt, n=cur_n, max_new_tokens=max_new_tokens)

        for s in gens:
            row = parse_sentence_to_features(s, feature_cols)
            synthetic_rows.append(row)

    # Truncate in case we slightly over-shot
    synthetic_rows = synthetic_rows[:N]
    return pd.DataFrame(synthetic_rows, columns=feature_cols)


In [ ]:
import re
import math
import numpy as np

def parse_sentence_to_features(text: str, feature_cols):
    """
    Parse 'col is val' pairs from generated text into a dict or list
    aligned with feature_cols (including 'target').
    """
    pattern = r'(\w+)\s+is\s+([^\s,\.]+)'
    matches = re.findall(pattern, text)

    values = {col: np.nan for col in feature_cols}

    for col, val in matches:
        if col not in values:
            continue

        # convert to bool if target
        if col == "target":
            if val.lower() == "true":
                values[col] = True
            elif val.lower() == "false":
                values[col] = False
            else:
                # malformed target → leave as NaN or None
                values[col] = None
            continue

        # numeric conversion for other columns
        try:
            if val.lower() == "none":
                values[col] = None
            elif "." in val:
                values[col] = float(val)
            else:
                values[col] = int(val)
        except ValueError:
            # fallback: keep raw string if it can't be parsed
            values[col] = val

    # Return in column order
    return [values[c] for c in feature_cols]


In [ ]:
# Last column is already renamed to 'target'
# df.columns: [..., 'f_hour_cos', 'target']

feature_cols = list(df.columns)  # this time INCLUDE 'target'

D_fake_joint = sampling_phase_joint(df, feature_cols, N=1000)

print(D_fake_joint.head())
print(D_fake_joint["target"].value_counts(dropna=False))
print(D_fake_joint.shape)


   creat_type_cd  f_cat_uniq  f_refresh_sum  slot_id  f_rows  f_up_sum  \
0             10          47            514       16     144       542   
1              8          26            682       54     121       586   
2              8          43            686       17     114       576   
3              8          47            818       50     136       769   
4              8          41            614       58      96       463   

   f_dislike_sum  f_refresh_mean  u_refreshTimes  u_newsCatInterestsST_len  \
0            619               9               9                         5   
1            573               6               6                         5   
2            574               7               7                         5   
3            762               9               9                         5   
4            372               9               9                         5   

   ...  adv_id  task_id  inter_type_cd  hispace_app_tags  spread_app_id  \
0  ...   12

Still fails, let's try something else